## Problem Description

**Goal:** Create a Python utility to manage conversational history by trimming messages based on a `max_tokens` budget, using `tiktoken` for local token counting.

**Constraints & Requirements:**

- Use `tiktoken` with the `cl100k_base` encoding.
- Do NOT call the Anthropic messages API inside the utility function.
- The function receives a list of message dicts (each with "role" and "content" string keys) and an integer `max_tokens` budget.
- If the conversation already fits within `max_tokens`, return it unchanged.
- If it exceeds the limit, remove the oldest message (index 0) and recount, repeating until it fits.
- Return a tuple: `(trimmed_messages, final_token_count)`.

**Input Schema:** `messages` is a list of dictionaries, where each dictionary has "role" (string) and "content" (string) keys.

In [10]:
import tiktoken


def trim_conversation(messages: list[dict], max_tokens: int) -> tuple[list[dict], int]:
    """
    Trim conversation history so total tokens fit within max_tokens.

    Args:
        messages: List of dicts with keys:
                  {"role": str, "content": str}
        max_tokens: Maximum allowed token budget

    Returns:
        (trimmed_messages, final_token_count)
    """

    # Step 1: set up the cl100k_base encoder
    encoder = tiktoken.get_encoding("cl100k_base")

    # Step 2: helper to count tokens across all messages
    # Count both role and content so the full message payload is represented
    def count_tokens(msgs: list[dict]) -> int:
        total = 0
        for msg in msgs:
            total += len(encoder.encode(msg["role"]))
            total += len(encoder.encode(msg["content"]))
        return total

    # Work on a copy so original input isn't modified
    trimmed_messages = messages.copy()

    # Initial token count
    token_count = count_tokens(trimmed_messages)

    # Step 3: remove oldest messages until within budget
    while token_count > max_tokens and trimmed_messages:
        trimmed_messages.pop(0)   # remove oldest message
        token_count = count_tokens(trimmed_messages)

    # Step 4: return result
    return trimmed_messages, token_count

### Example Usage

In [11]:
# Inlined Sample Data & Inputs:
sample_conversation = [
    {"role": "user",      "content": "What is the capital of France?"},
    {"role": "assistant", "content": "Paris is the capital of France."},
    {"role": "user",      "content": "How does photosynthesis work?"},
    {"role": "assistant", "content": "Photosynthesis converts sunlight into glucose using chlorophyll."},
    {"role": "user",      "content": "Can you summarise what we discussed?"},
]

max_tokens_budget = 40

trimmed_messages, final_token_count = trim_conversation(sample_conversation, max_tokens=max_tokens_budget)

print(f"Original messages: {len(sample_conversation)}")
print(f"Trimmed messages remaining: {len(trimmed_messages)}")
print(f"Final token count: {final_token_count}")
print("\nTrimmed Conversation:")
for msg in trimmed_messages:
    print(f"  Role: {msg['role']}, Content: {msg['content']}")

Original messages: 5
Trimmed messages remaining: 4
Final token count: 36

Trimmed Conversation:
  Role: assistant, Content: Paris is the capital of France.
  Role: user, Content: How does photosynthesis work?
  Role: assistant, Content: Photosynthesis converts sunlight into glucose using chlorophyll.
  Role: user, Content: Can you summarise what we discussed?
